# Timeouts
Bound how long an external operation may delay a request.


In [ ]:
# A timeout bounds how long one dependency may occupy a request.
import asyncio

async def slow_call() -> str:
    await asyncio.sleep(0.02)
    return "ok"

try:
    async with asyncio.timeout(0.01):
        await slow_call()
except TimeoutError:
    print("timed out")


## Polished version
Translate low-level timeout errors into a domain-specific failure at the provider boundary.


In [ ]:
# Translate asyncio's timeout into an error meaningful to application code.
from typing import Protocol

class Provider(Protocol):
    async def complete(self, prompt: str) -> str: ...

class ProviderTimedOut(Exception):
    pass

class SlowProvider:
    async def complete(self, prompt: str) -> str:
        await asyncio.sleep(0.02)
        return prompt.upper()

class TimedProvider:
    def __init__(self, provider: Provider, timeout_seconds: float) -> None:
        self.provider = provider
        self.timeout_seconds = timeout_seconds

    async def complete(self, prompt: str) -> str:
        try:
            async with asyncio.timeout(self.timeout_seconds):
                return await self.provider.complete(prompt)
        except TimeoutError as error:
            # Preserve the original cause while exposing a stable domain error.
            raise ProviderTimedOut from error

try:
    await TimedProvider(SlowProvider(), 0.01).complete("hello")
except ProviderTimedOut:
    print("provider timed out")


## Applied in this repository

The LLM [provider adapter](../00P2-project-llm-api/app/infrastructure/provider.py) configures an HTTP timeout and translates transport, status, and response-format failures into one application error.